# Demonstration/Testing of Loading Data from S3 Buckets

This notebook demonstrates loading data from the lake: **bronze** (raw Statcast dailies), **processed** (legacy by-player paths), **silver** (engineered `player_year_features.parquet`), and **gold** (model-ready `player_year_features_preprocessed.parquet` after silver→gold preprocessing).

## Imports

In [1]:
import sys, botocore, aiobotocore
print("executable:", sys.executable)
print("botocore:", botocore.__version__)
print("aiobotocore:", aiobotocore.__version__)

executable: c:\Users\bryan\Developer\Diamond-DNA\venv\Scripts\python.exe
botocore: 1.42.70
aiobotocore: 3.3.0


In [10]:
import pandas as pd
import s3fs
from collections import defaultdict
import re
from datetime import date, timedelta

pd.set_option('display.max_columns', None)

## Loading Data from Bronze Data Bucket

### Statcast

In [ ]:
# Building our key
BUCKET = "diamond-dna"
prefix = "bronze/statcast"

# Date Selection
s_date = date(2025, 11, 1)

path = f"s3://{BUCKET}/{prefix}/year={s_date.year}/date={s_date.strftime('%Y-%m-%d')}/statcast_pitches.parquet"

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Tell pandas/pyarrow to use the S3 filesystem
df = pd.read_parquet(path, filesystem=fs)
df.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,year,date
0,FS,2025-11-01,92.1,-2.02,5.15,"Yamamoto, Yoshinobu",672386,808967,grounded_into_double_play,hit_into_play,<NA>,<NA>,<NA>,<NA>,6,"Alejandro Kirk grounds into a double play, sho...",W,R,R,TOR,LAD,X,6,ground_ball,0,2,2025,-0.48,0.37,0.52943,2.165539,665489,<NA>,680718,1,11,Bot,119.57,155.13,<NA>,<NA>,<NA>,<NA>,7.524095,-133.878517,-2.517287,-7.425629,29.845675,-27.341263,3.13,1.5,5,71.9,-24,92.2,1320,6.5,813024,669257,518692,808975,571970,605141,571771,681909,681624,54.03,0.041,0.029,0.0,1,0,0,2,99,3,Split-Finger,4,5,4,5,5,4,4,5,Standard,Standard,242,-0.373,-0.177,73.8,7.7,0.047,0.177,88.0,-1,-1,0.373,0.373,26,26,27,27,2,5,1,1,<NA>,<NA>,2.33,0.48,0.48,36.9,8.375233,-7.664043,17.979522,48.911447,37.014581,2025,2025-11-01
1,CU,2025-11-01,80.3,-1.7,5.45,"Yamamoto, Yoshinobu",672386,808967,None,called_strike,<NA>,<NA>,<NA>,<NA>,4,Called Strike,W,R,R,TOR,LAD,S,<NA>,None,0,1,2025,1.36,-1.25,-0.279088,1.962539,665489,<NA>,680718,1,11,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.56166,-116.768446,1.472301,12.095261,26.350181,-44.003006,2.915852,1.373793,<NA>,<NA>,<NA>,79.7,2970,6.4,813024,669257,518692,808975,571970,605141,571771,681909,681624,54.15,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,99,2,Curveball,4,5,4,5,5,4,4,5,Standard,Standard,42,-0.053,-0.073,<NA>,<NA>,<NA>,0.073,<NA>,-1,-1,0.426,0.426,26,26,27,27,2,5,1,1,<NA>,<NA>,4.85,-1.36,-1.36,48.1,<NA>,<NA>,<NA>,<NA>,<NA>,2025,2025-11-01
2,FC,2025-11-01,92.8,-1.91,5.24,"Yamamoto, Yoshinobu",672386,808967,None,foul,<NA>,<NA>,<NA>,<NA>,9,Foul,W,R,R,TOR,LAD,S,<NA>,None,0,0,2025,0.23,0.6,0.564538,1.843233,665489,<NA>,680718,1,11,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,5.856053,-135.072611,-4.233184,1.577981,29.102128,-24.072732,3.13,1.5,1,74.8,-53,93.2,2446,6.5,813024,669257,518692,808975,571970,605141,571771,681909,681624,54.01,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,99,1,Cutter,4,5,4,5,5,4,4,5,Standard,Standard,187,-0.039,-0.041,73.0,7.5,<NA>,0.041,88.0,-1,-1,0.465,0.465,26,26,27,27,2,5,1,1,<NA>,<NA>,2.04,-0.23,-0.23,39.1,14.162806,5.694338,22.383742,47.652657,23.055298,2025,2025-11-01
3,FS,2025-11-01,90.9,-1.97,5.26,"Yamamoto, Yoshinobu",680718,808967,walk,blocked_ball,<NA>,<NA>,<NA>,<NA>,14,Addison Barger walks.,W,L,R,TOR,LAD,B,<NA>,None,3,0,2025,-0.69,0.02,0.473954,0.831956,665489,<NA>,<NA>,1,11,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,7.631083,-132.145001,-5.243103,-9.709392,28.153507,-30.982919,3.764518,1.788782,<NA>,<NA>,<NA>,91.2,1523,6.5,813024,669257,518692,8

### Statcast Running

In [9]:
# Building key
prefix = "bronze/statcast_running"

# Year selection
year = "2025"

path = f"s3://{BUCKET}/{prefix}/year={year}/statcast_sprint_speed.parquet"

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Tell pandas/pyarrow to use the S3 filesystem
df = pd.read_parquet(path, filesystem=fs)
df.head()

,"last_name, first_name",player_id,team_id,team,position,age,competitive_runs,bolts,hp_to_1b,sprint_speed,year
0,"Turner, Trea",607208,143,PHI,SS,32,265,117.0,4.22,30.3,2025
1,"Scott II, Victor",687363,138,STL,CF,24,141,87.0,4.13,30.2,2025
2,"Witt Jr., Bobby",677951,118,KC,SS,25,245,101.0,4.15,30.2,2025
3,"Buxton, Byron",621439,142,MIN,CF,31,165,56.0,4.13,30.2,2025
4,"Hill, Derek",656537,145,CWS,CF,29,50,24.0,4.21,30.1,2025


## Loading Data from Silver

### Pitcher

In [ ]:
# Building our key (legacy by-player pitch parquet path; current pipeline uses bronze dailies + silver feature tables)
BUCKET = "diamond-dna"
prefix = "silver"
role = 'pitcher'

# Year selection
year = '2025'

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

# Read as a single file stream so PyArrow doesn't discover Hive partitioning.
# (Partition discovery would add a dictionary-encoded "year" that conflicts with the int32 "year" column inside the file.)
path = f"s3://{BUCKET}/{prefix}/pitcher/year={year}/player_year_features.parquet"
fs = s3fs.S3FileSystem()

with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)
    
df.head()

,role,player_id,year,n_pitches_total,batter_swing_rate,batter_zone_swing_rate,batter_chase_rate,batter_contact_rate,batter_whiff_rate,in_zone_rate,release_speed_max,fastball_velo_mean,offspeed_velo_mean,velo_differential,release_speed_iqr,release_spin_rate_iqr,pfx_x_iqr,release_extension_mean,release_extension_iqr,pfx_z_mean,pfx_z_iqr,plate_x_mean,plate_x_sd,plate_z_mean,plate_z_sd,edge_percent,meatball_percent,first_pitch_strike_rate,xwoba_allowed_lhb_mean,xwoba_allowed_rhb_mean,platoon_xwoba_allowed_diff,gb_percent_allowed,ld_percent_allowed,fb_percent_allowed,iffb_percent_allowed,delta_run_exp_mean,pt_CH_release_speed_mean,pt_CH_release_spin_rate_mean,pt_CH_pfx_x_mean,pt_FC_release_speed_mean,pt_FC_release_spin_rate_mean,pt_FC_pfx_x_mean,pt_CU_release_speed_mean,pt_CU_release_spin_rate_mean,pt_CU_pfx_x_mean,pt_SI_release_speed_mean,pt_SI_release_spin_rate_mean,pt_SI_pfx_x_mean,pt_FF_release_speed_mean,pt_FF_release_spin_rate_mean,pt_FF_pfx_x_mean,pt_NONE_release_speed_mean,pt_NONE_release_spin_rate_mean,pt_NONE_pfx_x_mean,pt_CS_release_speed_mean,pt_CS_release_spin_rate_mean,pt_CS_pfx_x_mean,pitch_type_CU_share,pitch_type_SI_share,pitch_type_FC_share,pitch_type_FF_share,pitch_type_CH_share,pitch_type_CS_share,pitch_type_SL_share,pitch_type_entropy,pt_SL_release_speed_mean,pt_SL_release_spin_rate_mean,pt_SL_pfx_x_mean,pt_ST_release_speed_mean,pt_ST_release_spin_rate_mean,pt_ST_pfx_x_mean,pitch_type_ST_share,pitch_type_PO_share,pt_KC_release_speed_mean,pt_KC_release_spin_rate_mean,pt_KC_pfx_x_mean,pt_FS_release_speed_mean,pt_FS_release_spin_rate_mean,pt_FS_pfx_x_mean,pitch_type_FS_share,pitch_type_KC_share,pitch_type_SV_share,pt_SV_release_speed_mean,pt_SV_release_spin_rate_mean,pt_SV_pfx_x_mean,pitch_type_FA_share,pitch_type_EP_share,pitch_type_UN_share,pitch_type_FO_share,pt_SC_release_speed_mean,pt_SC_release_spin_rate_mean,pt_SC_pfx_x_mean,pitch_type_SC_share,pt_KN_release_speed_mean,pt_KN_release_spin_rate_mean,pt_KN_pfx_x_mean,pitch_type_KN_share,pt_FO_release_speed_mean,pt_FO_release_spin_rate_mean,pt_FO_pfx_x_mean
0,pitcher,434378,2025,2775,0.483243,0.336577,0.146667,0.376937,0.106306,0.491892,98.3,93.926526,83.441730,10.484796,9.9,190.75,1.2700,6.016940,0.3,0.721243,1.300,0.113814,0.774437,2.469420,0.992253,0.833700,0.166300,0.623762,0.373694,0.361138,0.012556,0.357430,0.251004,0.303213,0.088353,-0.000224,84.695614,1723.048246,-1.079561,NaN,NaN,NaN,78.457403,2746.644156,0.625481,NaN,NaN,NaN,93.931911,2432.033634,-0.739491,NaN,NaN,NaN,NaN,NaN,NaN,0.143336,0.003723,NaN,0.453835,0.084885,NaN,0.231943,1.411599,87.108186,2499.539326,0.339021,80.495475,2662.352941,0.965656,0.082278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,pitcher,445276,2025,925,0.550270,0.427027,0.123243,0.432432,0.117838,0.563243,96.9,92.750964,83.039535,9.711429,2.6,153.50,0.3100,6.880131,0.2,1.405131,0.190,-0.123323,0.655352,2.673284,0.961436,0.815739,0.184261,0.627530,0.390560,0.337013,0.053547,0.303030,0.163636,0.418182,0.115152,-0.011016,NaN,NaN,NaN,92.710858,2592.426273,0.197212,NaN,NaN,NaN,93.107143,2320.154762,-0.659643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.091703,0.814410,NaN,NaN,NaN,0.060044,0.669764,83.696364,2456.709091,0.602727,81.874194,2509.290323,1.034516,0.033843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pitcher,450203,2025,2605,0.467179,0.322457,0.144722,0.351248,0.115931,0.479463,97.3,93.068385,82.690887,10.377498,12.2,879.50,2.3600,6.155416,0.3,0.065130,1.560,0.082078,0.896490,2.236497,0.940283,0.865492,0.134508,0.596096,0.377818,0.379542,-0.001724,0.450704,0.251174,0.246479,0.051643,0.010534,87.516206,1815.205534,-1.140514,87.996522,2513.904348,0.072565,81.440061,3163.086066,1.280010,94.013506,2219.574026,-1.480130,94.208523,2322.633523,-0.995270,NaN,NaN,NaN,NaN,NaN,NaN,0.383046,0.151099,0.090267,0.276295,0.099294,NaN,NaN,1.454942,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

### Batter

In [16]:
prefix = 'silver'
role = 'batter'

# Year selection
year = "2025"

# Use AWS creds to access S3 buckets
fs = s3fs.S3FileSystem()

path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_features.parquet"
with fs.open(path, "rb") as f:
    df = pd.read_parquet(f)

pd.set_option('display.max_columns', None)
df.head()


,role,player_id,year,n_pitches_total,swing_rate,zone_swing_rate,chase_rate,contact_rate,whiff_rate,barrel_rate,hard_hit_rate,pull_percent,opposite_field_percent,gb_percent,ld_percent,fb_percent,iffb_percent,sweet_spot_percent,launch_speed_mean,launch_speed_iqr,launch_angle_mean,launch_angle_iqr,iso_value_mean,estimated_slg_using_speedangle_mean,woba_value_mean,estimated_woba_using_speedangle_mean,sprint_speed_mean,def_oaa_total,def_actual_fielding_success_rate_mean,def_adj_estimated_fielding_success_rate_mean,def_outfield_catch_completion_rate,def_arm_strength_max_mph,def_pop_time_2b_sec,def_framing_runs,def_drs_total
0,batter,456781,2025,774,0.512920,0.326873,0.186047,0.391473,0.121447,0.010830,0.223827,0.379310,0.282759,0.482759,0.248276,0.200000,0.068966,0.306859,83.162816,19.400,17.787004,42.00,0.095000,0.434493,0.301250,0.258718,25.0,1.0,0.79,0.77,NaN,72.1,NaN,NaN,NaN
1,batter,457705,2025,2344,0.453925,0.349829,0.104096,0.342577,0.111348,0.016598,0.257261,0.419355,0.182796,0.428954,0.219839,0.270777,0.080429,0.295172,83.360719,21.850,16.840000,45.00,0.106830,0.537488,0.307005,0.333453,27.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,batter,457759,2025,822,0.479319,0.336983,0.142336,0.403893,0.075426,0.006623,0.168874,0.321678,0.258741,0.384615,0.244755,0.286713,0.083916,0.294702,82.212252,18.075,24.238411,38.75,0.080808,0.467290,0.279040,0.298232,25.3,NaN,NaN,NaN,NaN,75.7,NaN,NaN,NaN
3,batter,467793,2025,2078,0.445621,0.321944,0.123677,0.345043,0.100577,0.007669,0.263804,0.502907,0.151163,0.409884,0.220930,0.261628,0.107558,0.279141,83.118558,23.200,18.113497,52.25,0.095808,0.447878,0.289421,0.290137,25.6,8.0,0.78,0.75,NaN,87.1,NaN,NaN,NaN
4,batter,500743,2025,1267,0.451460,0.315706,0.135754,0.376480,0.074980,0.015766,0.213964,0.368030,0.226766,0.457249,0.241636,0.234201,0.066914,0.295964,81.318694,20.650,14.941704,45.00,0.122093,0.434925,0.315407,0.296853,25.9,7.0,0.81,0.78,NaN,86.8,NaN,NaN,NaN


## Loading gold (preprocessed) player-year feature tables

In [18]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"  # default GOLD_PREFIX in src/pipeline/settings.py

role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_features_preprocessed.parquet"
with fs.open(path, "rb") as f:
    df_gold = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df_gold.head()

,role,player_id,year,n_pitches_total,swing_rate,zone_swing_rate,chase_rate,contact_rate,whiff_rate,barrel_rate,hard_hit_rate,pull_percent,opposite_field_percent,gb_percent,ld_percent,fb_percent,iffb_percent,sweet_spot_percent,launch_speed_mean,launch_speed_iqr,launch_angle_mean,launch_angle_iqr,iso_value_mean,estimated_slg_using_speedangle_mean,estimated_woba_using_speedangle_mean,sprint_speed_mean,def_oaa_total,def_adj_estimated_fielding_success_rate_mean,def_outfield_catch_completion_rate,def_arm_strength_max_mph,def_pop_time_2b_sec,def_framing_runs,def_drs_total,def_framing_runs_was_missing,def_pop_time_2b_sec_was_missing
0,batter,456781,2025,774,0.512920,0.326873,0.186047,0.391473,0.121447,0.010830,0.223827,0.379310,0.282759,0.482759,0.248276,0.200000,0.068966,0.306859,83.162816,19.400,17.787004,42.00,0.095000,0.434493,0.258718,25.0,1.0,0.77,0.0,72.1,0.0,0.0,0.0,1,1
1,batter,457705,2025,2344,0.453925,0.349829,0.104096,0.342577,0.111348,0.016598,0.257261,0.419355,0.182796,0.428954,0.219839,0.270777,0.080429,0.295172,83.360719,21.850,16.840000,45.00,0.106830,0.537488,0.333453,27.1,0.0,0.00,0.0,0.0,0.0,0.0,0.0,1,1
2,batter,457759,2025,822,0.479319,0.336983,0.142336,0.403893,0.075426,0.006623,0.168874,0.321678,0.258741,0.384615,0.244755,0.286713,0.083916,0.294702,82.212252,18.075,24.238411,38.75,0.080808,0.467290,0.298232,25.3,0.0,0.00,0.0,75.7,0.0,0.0,0.0,1,1
3,batter,467793,2025,2078,0.445621,0.321944,0.123677,0.345043,0.100577,0.007669,0.263804,0.502907,0.151163,0.409884,0.220930,0.261628,0.107558,0.279141,83.118558,23.200,18.113497,52.25,0.095808,0.447878,0.290137,25.6,8.0,0.75,0.0,87.1,0.0,0.0,0.0,1,1
4,batter,500743,2025,1267,0.451460,0.315706,0.135754,0.376480,0.074980,0.015766,0.213964,0.368030,0.226766,0.457249,0.241636,0.234201,0.066914,0.295964,81.318694,20.650,14.941704,45.00,0.122093,0.434925,0.296853,25.9,7.0,0.78,0.0,86.8,0.0,0.0,0.0,1,1


In [ ]:
import json

meta_path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/preprocessing_metadata.json"
try:
    with fs.open(meta_path, "r") as f:
        meta = json.load(f)
    print("Top-level keys:", sorted(meta.keys()))
    print("Row count:", meta.get("row_count"))
    print("Feature columns:", len(meta.get("feature_columns", [])))
except FileNotFoundError:
    print(f"No metadata at {meta_path} (run silver→gold preprocessing for this role/year first).")

Top-level keys: ['correlation_dropped_columns', 'correlation_threshold', 'dropped_columns', 'feature_columns', 'generated_at_utc', 'hard_dropped_columns', 'id_columns', 'missing_indicator_columns', 'near_zero_variance_dropped_columns', 'numeric_columns_filled_zero', 'role', 'row_count', 'year']
Row count: 392
Feature columns: 32
